# Monterey Bay, Inside Out

Day 15 · Inside out. The sea floor under Monterey Bay turns over, so every metre of depth becomes a metre of height.
The land stays where it is. Run `python scripts\fetch_week3.py day15` first.

## 1 · Settings

In [ ]:
DAY      = 15
PROMPT   = "Inside out"
SLUG     = "insideout-monterey-canyon"
NOTEBOOK = "day15_insideout_monterey_canyon.ipynb"
TITLE    = "Monterey Bay,\nInside Out"
CREDITS  = "Bathymetry and land elevation: NOAA NCEI DEM Global Mosaic (Coastal Relief Model), via the NCEI image service."

VE       = 5        # height exaggeration, said on the map
STEP     = 3        # use every 3rd grid point (1 = full detail, slower to render)
FPS      = 24
SECONDS  = dict(still=1.5, flip=3.5, hold=3.5)      # still under the sea, the flip, then a slow turn to finish
CAMERA   = dict(elev=27, azim_start=18, azim_end=-8)

PAGE, INK, SOFT = "#f2efe8", "#14202a", "#6f7a80"
TITLE_FONT, TEXT_FONT, TEXT_BOLD = "OLD", "LATO", "LATO_BOLD"     # IM Fell English (old sea charts) + Lato, from mapkit/poster.py
PLACES   = {"Santa Cruz": (36.974, -122.030), "Monterey": (36.600, -121.894)}   # labels on the land, lat/lon
DEEP = ["#0b2a4a", "#1f5f8b", "#5aa6c8", "#c8e8f0"]       # deepest to shallowest
LAND = ["#ddd0b5", "#b9a888", "#8a7b62"]                  # lowest to highest

## 2 · Load

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LightSource, LinearSegmentedColormap
from mapkit import poster, publish
from mapkit.paths import PROCESSED

d = np.load(PROCESSED / "monterey_bathy.npz")
z_full = d["z"].astype(float)
z_full[z_full == d["nodata"]] = np.nan
w, s, e, n = d["bounds"]
z = z_full[::STEP, ::STEP]
H, W = z.shape
lat0 = (s + n) / 2
xkm = np.linspace(0, (e - w) * 111.32 * np.cos(np.radians(lat0)), W)     # metres on the ground, east-west
ykm = np.linspace((n - s) * 110.57, 0, H)                                 # north-south
X, Y = np.meshgrid(xkm, ykm)
sea = z < 0
print(f"{W} x {H} points, {xkm[-1]:.0f} x {ykm[0]:.0f} km")

## 3 · Analysis, every number on the map comes from here

In [ ]:
DEEPEST = -np.nanmin(z_full)          # from the full-detail grid, not the thinned one
HIGHEST = np.nanmax(z_full)
SEA_SHARE = (z_full < 0).sum() / np.isfinite(z_full).sum() * 100
print(f"Deepest point in the frame: {DEEPEST:,.0f} m below sea level")
print(f"Highest land in the frame:  {HIGHEST:,.0f} m")
print(f"Turned over, the deepest point stands {DEEPEST / HIGHEST:.1f}x taller than the highest land")
print(f"Sea covers {SEA_SHARE:.0f}% of the frame")

## 4 · Map, one frame at a time

In [ ]:
deep_cmap = LinearSegmentedColormap.from_list("deep", DEEP)
land_cmap = LinearSegmentedColormap.from_list("land", LAND)
light = LightSource(azdeg=315, altdeg=40)
F = poster.font
TF, TX, TB = (getattr(poster, k) for k in (TITLE_FONT, TEXT_FONT, TEXT_BOLD))

def colours(z_now):
    """Colour by the ORIGINAL depth, so the canyon is still blue once it becomes a ridge. Shade by the current shape."""
    rgba = np.ones(z.shape + (4,))
    rgba[sea] = deep_cmap(1 - np.clip(-z[sea] / DEEPEST, 0, 1))
    rgba[~sea] = land_cmap(np.clip(z[~sea] / HIGHEST, 0, 1))
    shade = light.hillshade(z_now, vert_exag=VE / 1000, dx=xkm[1] - xkm[0], dy=ykm[0] - ykm[1])
    rgba[..., :3] = (rgba[..., :3] * (0.55 + 0.6 * shade[..., None])).clip(0, 1)
    return rgba

def ease(t):
    return 0.5 - 0.5 * np.cos(np.pi * np.clip(t, 0, 1))

def draw(flip, azim):
    """flip: 0 = the real sea floor, 1 = fully inside out."""
    z_now = np.where(sea, z * (1 - 2 * flip), z)            # depth d becomes height d
    fig = plt.figure(figsize=(10.8, 13.5), dpi=100, facecolor=PAGE)
    ax = fig.add_axes([-0.2, 0.06, 1.4, 0.80], projection="3d", facecolor=PAGE)
    ax.plot_surface(X, Y, z_now / 1000 * VE, facecolors=colours(z_now), rstride=1, cstride=1,
                    lw=0, antialiased=False, shade=False)
    ax.plot([0, xkm[-1], xkm[-1], 0, 0], [0, 0, ykm[0], ykm[0], 0], [0] * 5, color=INK, lw=0.7, ls=(0, (3, 3)))
    top = DEEPEST / 1000 * VE
    ax.set_xlim(0, xkm[-1]); ax.set_ylim(0, ykm[0]); ax.set_zlim(-top, top)
    ax.set_box_aspect((xkm[-1], ykm[0], 2 * top * 0.9), zoom=1.25)
    ax.view_init(elev=CAMERA["elev"], azim=azim)
    ax.set_axis_off()
    for name, (la, lo) in PLACES.items():
        px = (lo - w) * 111.32 * np.cos(np.radians(lat0)); py = (la - s) * 110.57
        ax.text(px, py, 0.7 * VE, name, color=INK, fontproperties=F(TB, 12.5), ha="center", zorder=10)

    poster.title(fig, TITLE, x=0.06, y=0.955, size=56, color=INK, path=TF, line=1.0)
    fig.text(0.06, 0.835, "The sea floor turns over: every metre of depth becomes a metre of height.",
             color=SOFT, fontproperties=F(TX, 15), va="top")
    # the live counter, deepest point in the frame
    now = round(-DEEPEST * (1 - 2 * flip))
    word = "below sea level" if now < 0 else ("above sea level" if now > 0 else "at sea level")
    fig.text(0.94, 0.955, f"{abs(now):,.0f} m" if now >= 0 else f"-{abs(now):,.0f} m", color=DEEP[1] if now < 0 else INK,
             fontproperties=F(TF, 44), ha="right", va="top")
    fig.text(0.94, 0.905, word, color=SOFT, fontproperties=F(TX, 13.5), ha="right", va="top")
    fig.text(0.94, 0.885, "the deepest point in the frame", color=SOFT, fontproperties=F(TX, 11.5), ha="right", va="top")
    fig.text(0.06, 0.11, f"Height exaggerated {VE}x · dashed line: sea level · the land does not move",
             color=SOFT, fontproperties=F(TX, 12))
    poster.credit(fig, "Elevation and depth: NOAA NCEI Coastal Relief Model (DEM Global Mosaic)",
                  x=0.06, y=0.03, color=SOFT, path=TX, name_path=TB, name_color=INK)
    return fig

def frames(fps=FPS):
    n_still, n_flip, n_hold = (int(SECONDS[k] * fps) for k in ("still", "flip", "hold"))
    a0, a1 = CAMERA["azim_start"], CAMERA["azim_end"]
    for i in range(n_still):
        yield 0.0, a0
    for i in range(n_flip):
        yield ease(i / (n_flip - 1)), a0
    for i in range(n_hold):
        yield 1.0, a0 + (a1 - a0) * ease(i / (n_hold - 1))

fig = draw(1.0, CAMERA["azim_start"])      # preview the last moment
plt.show()

## 5 · Export: MP4 for LinkedIn, plus a still for the gallery
Takes 5 to 15 minutes (it draws about 200 frames). Needs imageio-ffmpeg (`pip install imageio-ffmpeg`, already there from the Day 1 video).
Quick test first? Set FPS = 8 in Settings, then put it back to 24 for the real one.

In [ ]:
import io, imageio.v2 as imageio
from PIL import Image
from mapkit.paths import OUTPUT

mp4 = OUTPUT / f"day{DAY:02d}_{SLUG}.mp4"
with imageio.get_writer(mp4, fps=FPS, codec="libx264", quality=None, pixelformat="yuv420p",
                        ffmpeg_params=["-crf", "18", "-preset", "veryfast"]) as out:
    for k, (flip, azim) in enumerate(frames()):
        f = draw(flip, azim)
        buf = io.BytesIO(); f.savefig(buf, format="png", facecolor=PAGE); plt.close(f)
        out.append_data(np.array(Image.open(buf).convert("RGB")))
        if k % 24 == 0:
            print(f"frame {k}")
print("saved", mp4)

still = draw(1.0, CAMERA["azim_start"])
publish.save(still, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"Turned over, the deepest point ({DEEPEST:,.0f} m) stands {DEEPEST / HIGHEST:.1f}x taller than the highest land",
             series="city", tool="Python", notebook=NOTEBOOK, credits=CREDITS)